# Assignment 3 Module A: Transaction Engine Tests

Hey! This notebook shows that my custom transaction engine works on top of the B+ tree database from assignment 2.

What I am testing here:
- **Cell 2**: Atomicity (if something breaks midway, it rolls back completely)
- **Cell 3**: Durability (after reboot, committed data is rebuilt from WAL)
- **Cell 4**: Multi-table rollback (one transaction touching 3 tables)
- **Cell 5**: Structural consistency checks for B+ tree tables
- **Cell 6**: Referential integrity check (PersonVisit.MemberID -> Member.MemberID)
- **Cell 7**: Isolation check with two concurrent threads (serialized execution)

Let's go :)

In [11]:
# setting up the db and transaction manager...
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from database.db_manager  import DatabaseManager
from database.wal         import WALWriter
from database.transaction import TransactionManager
from database.recovery    import RecoveryManager
from database.consistency import ConsistencyChecker

os.makedirs("logs", exist_ok=True)
WAL_PATH = "logs/test_log.jsonl"

# starting fresh so deleting old log if it exists
if os.path.exists(WAL_PATH):
    os.remove(WAL_PATH)
    print(f"deleted old log file: {WAL_PATH}")

db  = DatabaseManager.create_gateguard_db()
wal = WALWriter(WAL_PATH)
tm  = TransactionManager(db, wal)
cc  = ConsistencyChecker(db)

print("\nsetup done!")
print(f"using db: {db}")

deleted old log file: logs/test_log.jsonl
[DB] Created table 'MemberType' (pk='TypeID', order=4)
[DB] Created table 'Member' (pk='MemberID', order=4)
[DB] Created table 'VehicleType' (pk='TypeID', order=4)
[DB] Created table 'Vehicle' (pk='VehicleID', order=4)
[DB] Created table 'Gate' (pk='GateID', order=4)
[DB] Created table 'GateOccupancy' (pk='OccupancyID', order=4)
[DB] Created table 'Role' (pk='RoleID', order=4)
[DB] Created table 'User' (pk='UserID', order=4)
[DB] Created table 'PersonVisit' (pk='VisitID', order=4)
[DB] Created table 'VehicleVisit' (pk='VVVisitID', order=4)

setup done!
using db: DatabaseManager(name='GateGuardDB', tables=['MemberType', 'Member', 'VehicleType', 'Vehicle', 'Gate', 'GateOccupancy', 'Role', 'User', 'PersonVisit', 'VehicleVisit'])


In [12]:
# -----------------------------------------------------
# Test 1: ATOMICITY
# gonna insert a member and then simulate a crash
# before the visit gets inserted, so it should rollback.
# -----------------------------------------------------
print("running atomicity test...")

MEMBER_ID = 999

try:
    with tm.transaction("test_atomicity") as txn:
        txn.insert("Member", {
            "MemberID": MEMBER_ID, "Name": "Test User",
            "Email": "test@iitgn.ac.in", "ContactNumber": "1234567890",
            "Image": None, "Age": 20, "Department": "CS", "TypeID": 1,
            "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
        })
        print(f"okay inserted member {MEMBER_ID} inside transaction...")
        
        # oops simulating a crash here lol
        raise RuntimeError("oh no the network disconnected!!")
        
        # this part wont run
        txn.insert("PersonVisit", {"VisitID": 999, "MemberID": 999})

except RuntimeError as e:
    print(f"caught the crash: {e}")

# checking if member 999 is actually gone
check_member = db.get_table("Member").select(MEMBER_ID)
assert check_member is None, f"fail! member {MEMBER_ID} is still there"
print(f"\nsuccess!! member {MEMBER_ID} is not in the db because it rolled back.")

cc.check_all(verbose=False)

running atomicity test...
starting txn: txn-test_atomicity-a1fe6838...
  -> inserting into Member with key 999
okay inserted member 999 inside transaction...
\noops transaction crashed: oh no the network disconnected!!
auto rolling back now...
uh oh rolling back txn txn-test_atomicity-a1fe6838! undoing 1 things..
  -> undoing insert on Member
txn aborted: txn-test_atomicity-a1fe6838
caught the crash: oh no the network disconnected!!

success!! member 999 is not in the db because it rolled back.


{'overall_ok': True,
 'tables': {'MemberType': {'table': 'MemberType',
   'ok': True,
   'record_count': 0,
   'issues': []},
  'Member': {'table': 'Member', 'ok': True, 'record_count': 0, 'issues': []},
  'VehicleType': {'table': 'VehicleType',
   'ok': True,
   'record_count': 0,
   'issues': []},
  'Vehicle': {'table': 'Vehicle', 'ok': True, 'record_count': 0, 'issues': []},
  'Gate': {'table': 'Gate', 'ok': True, 'record_count': 0, 'issues': []},
  'GateOccupancy': {'table': 'GateOccupancy',
   'ok': True,
   'record_count': 0,
   'issues': []},
  'Role': {'table': 'Role', 'ok': True, 'record_count': 0, 'issues': []},
  'User': {'table': 'User', 'ok': True, 'record_count': 0, 'issues': []},
  'PersonVisit': {'table': 'PersonVisit',
   'ok': True,
   'record_count': 0,
   'issues': []},
  'VehicleVisit': {'table': 'VehicleVisit',
   'ok': True,
   'record_count': 0,
   'issues': []}}}

In [13]:
# -----------------------------------------------------
# Test 2: DURABILITY
# gonna commit something real, then 'reboot' the DB
# to see if it reads from the log file correctly.
# -----------------------------------------------------
print("running durability test...")

D_MEMBER_ID = 1
D_VISIT_ID  = 1
D_OCC_ID    = 1

with tm.transaction("test_durability") as txn:
    txn.insert("Member", {
        "MemberID": D_MEMBER_ID, "Name": "Pushkar Modi",
        "Email": "pushkar@iitgn.ac.in", "ContactNumber": "0000000000",
        "Image": None, "Age": 22, "Department": "CS", "TypeID": 1,
        "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
    })
    txn.insert("PersonVisit", {
        "VisitID": D_VISIT_ID, "MemberID": D_MEMBER_ID,
        "EntryGateID": 1, "ExitGateID": None,
        "EntryTime": "2026-03-26T08:00:00", "ExitTime": None, "VehicleID": None
    })
    txn.insert("GateOccupancy", {
        "OccupancyID": D_OCC_ID, "GateID": 1, "Timestamp": "2026-03-26T08:00:00",
        "PersonCount": 1, "VehicleCount": 0
    })

print("committed 1 member, 1 visit, and 1 occupancy record to the db.")

print("\nsimulating server reboot......")
db2  = DatabaseManager.create_gateguard_db()   # totally empty new db
rm   = RecoveryManager(db2, WAL_PATH)
rm.recover()

recovered_member = db2.get_table("Member").select(D_MEMBER_ID)
assert recovered_member is not None, "member not found after recovery!"
print(f"nice, recovered member: {recovered_member['Name']}")

recovered_visit = db2.get_table("PersonVisit").select(D_VISIT_ID)
assert recovered_visit is not None, "visit not found!"
print(f"and recovered visit id: {recovered_visit['VisitID']}")

recovered_occ = db2.get_table("GateOccupancy").select(D_OCC_ID)
assert recovered_occ is not None, "occupancy not found!"
print(f"and recovered occupancy id: {recovered_occ['OccupancyID']}")


running durability test...
starting txn: txn-test_durability-28bca229...
  -> inserting into Member with key 1
  -> inserting into PersonVisit with key 1
  -> inserting into GateOccupancy with key 1
committed txn txn-test_durability-28bca229 yay!
committed 1 member, 1 visit, and 1 occupancy record to the db.

simulating server reboot......
[DB] Created table 'MemberType' (pk='TypeID', order=4)
[DB] Created table 'Member' (pk='MemberID', order=4)
[DB] Created table 'VehicleType' (pk='TypeID', order=4)
[DB] Created table 'Vehicle' (pk='VehicleID', order=4)
[DB] Created table 'Gate' (pk='GateID', order=4)
[DB] Created table 'GateOccupancy' (pk='OccupancyID', order=4)
[DB] Created table 'Role' (pk='RoleID', order=4)
[DB] Created table 'User' (pk='UserID', order=4)
[DB] Created table 'PersonVisit' (pk='VisitID', order=4)
[DB] Created table 'VehicleVisit' (pk='VVVisitID', order=4)

  [RECOVERY] Scanning WAL for crash recovery...
  WAL records       : 8
  Committed txns    : 1
  Uncommitted t

In [14]:
# -----------------------------------------------------
# Test 3: MULTI-TABLE ROLLBACK
# testing if exception rolls back all 3 tables
# -----------------------------------------------------
print("running multi table atomicity...")

count_m = db.get_table("Member").count()
count_v = db.get_table("PersonVisit").count()
count_g = db.get_table("GateOccupancy").count()
print(f"records before -> members: {count_m}, visits: {count_v}, occupancy: {count_g}")

try:
    with tm.transaction("multi_fail") as txn:
        txn.insert("Member", {
            "MemberID": 200, "Name": "Temp User",
            "Email": "temp@iitgn.ac.in", "ContactNumber": "111",
            "Image": None, "Age": 21, "Department": "EE", "TypeID": 1,
            "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
        })
        txn.insert("PersonVisit", {
            "VisitID": 200, "MemberID": 200, "EntryGateID": 2,
            "ExitGateID": None, "EntryTime": "2026-03-26T09:00:00",
            "ExitTime": None, "VehicleID": None
        })
        txn.insert("GateOccupancy", {
            "OccupancyID": 200, "GateID": 2, "Timestamp": "2026-03-26T09:00:00",
            "PersonCount": 50, "VehicleCount": 0
        })
        raise ValueError("fake error to trigger rollback")

except ValueError as e:
    print(f"error caught: {e}")

after_m = db.get_table("Member").count()
after_v = db.get_table("PersonVisit").count()
after_g = db.get_table("GateOccupancy").count()

assert after_m == count_m, "member count changed when it shouldn't have"
assert after_v == count_v, "visit count changed!"
assert after_g == count_g, "occupancy count changed!"
print("\npassed! all 3 tables rolled back perfectly.")


running multi table atomicity...
records before -> members: 1, visits: 1, occupancy: 1
starting txn: txn-multi_fail-7dbc9930...
  -> inserting into Member with key 200
  -> inserting into PersonVisit with key 200
  -> inserting into GateOccupancy with key 200
\noops transaction crashed: fake error to trigger rollback
auto rolling back now...
uh oh rolling back txn txn-multi_fail-7dbc9930! undoing 3 things..
  -> undoing insert on GateOccupancy
  -> undoing insert on PersonVisit
  -> undoing insert on Member
txn aborted: txn-multi_fail-7dbc9930
error caught: fake error to trigger rollback

passed! all 3 tables rolled back perfectly.


In [15]:
# -----------------------------------------------------
# Test 4: CONSISTENCY CHECK
# just testing our B+ tree didn't get corrupted
# after doing inserts, updates and deletes.
# -----------------------------------------------------
print("running tree consistency check...")

with tm.transaction("bulk_mix") as txn:
    for i in range(10, 15):
        txn.insert("Member", {
            "MemberID": i, "Name": f"Student_{i}", "Email": f"s{i}@iitgn.ac.in",
            "ContactNumber": "999", "Image": None,
            "Age": 20, "Department": "CS", "TypeID": 1,
            "CreatedAt": "2026-03-26", "UpdatedAt": "2026-03-26"
        })

with tm.transaction("bulk_upd") as txn:
    txn.update("Member", 10, {"Age": 21})
    txn.delete("Member", 14)

res = cc.check_all(verbose=True)
if res["overall_ok"]:
    print("yay, B+ Tree is still consistent! all good.")
else:
    print("uh oh something broke in the tree :(")

running tree consistency check...
starting txn: txn-bulk_mix-ccf806b5...
  -> inserting into Member with key 10
  -> inserting into Member with key 11
  -> inserting into Member with key 12
  -> inserting into Member with key 13
  -> inserting into Member with key 14
committed txn txn-bulk_mix-ccf806b5 yay!
starting txn: txn-bulk_upd-fff2f2a7...
  -> updating Member with key 10
  -> deleting from Member with key 14
committed txn txn-bulk_upd-fff2f2a7 yay!

  CONSISTENCY CHECK
  Table                       Records  Status
  --------------------------------------------------------
  MemberType                        0  OK
  Member                            5  OK
  VehicleType                       0  OK
  Vehicle                           0  OK
  Gate                              0  OK
  GateOccupancy                     1  OK
  Role                              0  OK
  User                              0  OK
  PersonVisit                       1  OK
  VehicleVisit                      

In [16]:
# -----------------------------------------------------
# Test 5: REFERENTIAL INTEGRITY
# checks PersonVisit.MemberID points to a real Member
# -----------------------------------------------------
print("running referential integrity test...")

with tm.transaction("ref_setup") as txn:
    txn.insert("Member", {
        "MemberID": 300, "Name": "Krina Paghadar",
        "Email": "krina@iitgn.ac.in", "ContactNumber": "9898989898",
        "Image": None, "Age": 21, "Department": "CS", "TypeID": 1,
        "CreatedAt": "2026-04-12", "UpdatedAt": "2026-04-12"
    })
    txn.insert("PersonVisit", {
        "VisitID": 300, "MemberID": 300, "EntryGateID": 1,
        "ExitGateID": None, "EntryTime": "2026-04-12T10:00:00",
        "ExitTime": None, "VehicleID": None
    })

r1 = cc.check_referential_integrity("PersonVisit", "MemberID", "Member", "MemberID")
assert r1["ok"], "referential check should pass for valid data"
print("good start: all visits currently point to valid members.")

bad_visit = {
    "VisitID": 9999, "MemberID": 99999,
    "EntryGateID": 1, "ExitGateID": None,
    "EntryTime": "2026-04-12T11:00:00", "ExitTime": None, "VehicleID": None
}
db.get_table("PersonVisit").insert(bad_visit)
print("injected one orphan visit row to test the checker.")

r2 = cc.check_referential_integrity("PersonVisit", "MemberID", "Member", "MemberID")
assert not r2["ok"], "checker should detect orphan visit rows"
print(f"checker caught it: {len(r2['orphans'])} orphan row(s).")

db.get_table("PersonVisit").delete(9999)
r3 = cc.check_referential_integrity("PersonVisit", "MemberID", "Member", "MemberID")
assert r3["ok"], "cleanup failed, orphan still present"
print("cleaned the bad row and referential integrity is valid again.")

running referential integrity test...
starting txn: txn-ref_setup-f34febd9...
  -> inserting into Member with key 300
  -> inserting into PersonVisit with key 300
committed txn txn-ref_setup-f34febd9 yay!
good start: all visits currently point to valid members.
injected one orphan visit row to test the checker.
  [Warning] Referential integrity FAIL: PersonVisit.MemberID -> Member.MemberID
    Orphan: child pk=9999  fk_val=99999 not in Member
checker caught it: 1 orphan row(s).
cleaned the bad row and referential integrity is valid again.


In [ ]:
# -----------------------------------------------------
# Test 6: ISOLATION WITH CONCURRENT THREADS
# verifies serialized execution with two racing threads
# -----------------------------------------------------
print("running isolation test with two threads...")

import threading, time

barrier = threading.Barrier(2)
results = []

def run_thread_a():
    try:
        txn = tm.begin("isolation_a")
        print("thread A started transaction and is inserting member 500")
        txn.insert("Member", {
            "MemberID": 500, "Name": "Thread A User",
            "Email": "thread.a@iitgn.ac.in", "ContactNumber": "5000001",
            "Image": None, "Age": 22, "Department": "CS", "TypeID": 1,
            "CreatedAt": "2026-04-12", "UpdatedAt": "2026-04-12"
        })
        barrier.wait()
        time.sleep(0.05) 
        txn.commit()
        results.append(("A", "committed"))
        print("thread A committed successfully")
    except Exception as e:
        results.append(("A", f"error: {e}"))
        print(f"thread A got an unexpected error: {e}")

def run_thread_b():
    barrier.wait()
    try:
        txn = tm.begin("isolation_b")
        txn.insert("Member", {
            "MemberID": 500, "Name": "Thread B User",
            "Email": "thread.b@iitgn.ac.in", "ContactNumber": "5000002",
            "Image": None, "Age": 23, "Department": "EE", "TypeID": 1,
            "CreatedAt": "2026-04-12", "UpdatedAt": "2026-04-12"
        })
        txn.commit()
        results.append(("B", "committed"))
        print("thread B committed (this should not happen)")
    except Exception as e:
        results.append(("B", "blocked by serialization"))
        print(f"thread B blocked as expected: {str(e)[:90]}")

ta = threading.Thread(target=run_thread_a, daemon=True)
tb = threading.Thread(target=run_thread_b, daemon=True)
ta.start()
tb.start()
ta.join(timeout=10) 
tb.join(timeout=10)

print(f"thread results: {results}")

member_500 = db.get_table("Member").select(500)
print(f"member 500 present as: {member_500['Name'] if member_500 else 'not found'}")

committed = [r for r in results if r[1] == "committed"]
blocked = [r for r in results if "blocked" in r[1]]

assert len(committed) == 1 and committed[0][0] == "A", "isolation violated: both threads committed"
assert len(blocked) == 1 and blocked[0][0] == "B", "expected thread B to be blocked"

final = cc.check_all(verbose=False)
assert final["overall_ok"], "consistency check failed after isolation test"
print("isolation test passed and consistency is still good.")

running isolation test with two threads...
starting txn: txn-isolation_a-f4fa15bf...
thread A started transaction and is inserting member 500
  -> inserting into Member with key 500
thread B blocked as expected: TransactionManager already has an active transaction ('txn-isolation_a-f4fa15bf'). Commit 
committed txn txn-isolation_a-f4fa15bf yay!
thread A committed successfully
thread results: [('B', 'blocked by serialization'), ('A', 'committed')]
member 500 present as: Thread A User
isolation test passed and consistency is still good.
